# APIs Advanced: Working with Model APIs using `requests`

In earlier tutorials, you used `requests`, worked with JSON responses, wrote functions, looped over records, and saved data to files. Here, we use the same workflow with an AI API. The new part is that the API generates information from the input we send.

By the end, you will be able to:

1. Authenticate and send a request to an AI API.
2. Inspect and extract information from a JSON response.
3. Use functions and loops to enrich records.
4. Request structured output and prepare results for validation.
5. Save enriched records for later use.

We use `requests` first so the HTTP mechanics are visible. A Python SDK wraps these same requests; it is introduced briefly in the appendix.

## Choose your path

- **Basic path:** follow every section, run the starter snippets, and open the optional hints and solutions when needed.
- **Advanced path:** write your own solutions before opening the collapsible answers, then use the checklist to review your work.

Both paths cover the same concepts; the advanced path simply provides less scaffolding.

## Before you start

Create a tilburg.ai API key through the [tilburg.ai API platform](https://api-portal.tilburg.ai/) using your Tilburg University account. Store it in a `.env` file in the same folder as this notebook:

```text
TILBURG_AI_API_KEY=sk-XXXXXXXXXXXXXXXXXXXX
```

Never commit the `.env` file or print the full key. Install the required packages with:

```bash
pip install requests python-dotenv
```

# 1. Authenticate and make a first request

The Responses API endpoint is:

```text
POST https://api.tilburg.ai/v1/responses
```

We use `POST` because this API endpoint expects us to submit structured input for the server to process. The model, prompt, instructions, and generation settings are sent as JSON in the request body. `POST` does not necessarily mean that a permanent record is created: here, the server performs a computation and returns a result.

The `Authorization` header contains the API key as a Bearer token. First, run the starter code below.

**Install the packages**

You have used `requests` before. We also use `python-dotenv` to read the API key from a `.env` file.

In [ ]:
%pip install requests python-dotenv

**Load the API key**

An API key identifies you to the API. We store it in `.env` instead of writing it directly in the notebook. We display only its first 10 characters to check that it loaded.

In [ ]:
from dotenv import find_dotenv, load_dotenv
import os

found = find_dotenv(usecwd=True)
load_dotenv(found, override=True)
api_key = os.getenv("TILBURG_AI_API_KEY")

if not api_key:
    raise RuntimeError("TILBURG_AI_API_KEY was not found. Check your .env file.")

print("API key preview:", api_key[:10] + "...")

**Choose the endpoint**

An endpoint is a URL for one API task. The `/responses` endpoint generates a model response.

In [ ]:
responses_endpoint = "https://api.tilburg.ai/v1/responses"
print(responses_endpoint)

**Build the headers**

Headers are extra information sent with a request. `Authorization` identifies you, and `Content-Type` tells the server that the request body is JSON. `Bearer` means that the secret token follows the word `Bearer`.

In [ ]:
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

print("Headers prepared; the secret key itself is not printed.")

**Build the payload**

The payload is the JSON data sent to the model. It contains the model name, instructions, and input.

In [ ]:
payload = {
    "model": "gpt-5.4-nano",
    "instructions": (
        "Identify one relevant customer group and give one reason based only "
        "on the information in the question. Answer in two sentences."
    ),
    "input": "Which customer segment is most likely to respond to a discount campaign?",
    "temperature": 1
}

print(payload)

**Send the request**

`requests.post()` sends the payload to the endpoint. The status code tells us what happened: `200` usually means success, `401` often means an authentication problem, and `400` often means an invalid request.

In [ ]:
import requests
import json

response = requests.post(
    responses_endpoint,
    headers=headers,
    json=payload,
    timeout=60
)

print("Status code:", response.status_code)
print(json.dumps(response.json(), indent=2))

**Inspect the response**

The server sends JSON back. `.json()` converts it into Python dictionaries and lists. Inspect the result before extracting a field.

In [ ]:
import json

data = response.json()
print(data.keys())
print(json.dumps(data, indent=2))

### One request does not remember another

Each API request is independent. The model does not automatically remember an earlier request. If a later request needs previous information, include that information again in the new request.

For example, send the earlier message and the follow-up question together:

In [ ]:
import json

payload = {
    "model": "gpt-5.4-nano",
    "input": [
        {
            "role": "user",
            "content": "The customer is interested in the spring-sale campaign."
        },
        {
            "role": "user",
            "content": "Which campaign is the customer interested in?"
        }
    ],
    "temperature": 0
}

response = requests.post(
    responses_endpoint,
    headers=headers,
    json=payload,
    timeout=60
)
print("Status code:", response.status_code)
print(json.dumps(response.json(), indent=2))

### Exercise 1 — Change the request payload

Modify the starter snippet so that it asks:

> Which customers should receive a retention offer after an abandoned-cart event?

Then:

- Run the request.
- Inspect the status code.
- Inspect the returned JSON.
- Explain what the URL, headers, and `payload` each contribute to the request.

In [ ]:
import json

# Starter code — change only the input question.
payload = {
    "model": "gpt-5.4-nano",
    "instructions": "Answer in one or two sentences and identify one relevant customer group.",
    "input": "Which customer segment is most likely to respond to a discount campaign?",
    "temperature": 1
}

response = requests.post(
    responses_endpoint,
    headers=headers,
    json=payload,
    timeout=60
)
print("Status code:", response.status_code)
print(json.dumps(response.json(), indent=2))

<details>
<summary><b>Need a hint?</b></summary>

Keep `responses_endpoint`, `headers`, and `requests.post()` unchanged. Change only the relevant text in `payload["input"]`.

</details>

<details>
<summary><b>Solution — Exercise 1</b></summary>

```python
payload = {
    "model": "gpt-5.4-nano",
    "instructions": (
        "Identify one relevant customer group and give one reason based only "
        "on the information in the question. Answer in two sentences."
    ),
    "input": (
        "Which customers should receive a retention offer after an "
        "abandoned-cart event?"
    ),
    "temperature": 1
}

response = requests.post(
    responses_endpoint,
    headers=headers,
    json=payload,
    timeout=60
)
print("Status code:", response.status_code)
print(json.dumps(response.json(), indent=2))
```

This is a complete request: the payload, endpoint, headers, POST request, status code, and response are all included.

</details>

<div class="alert alert-block alert-success"><b>Key idea:</b><br>An API request combines an endpoint, an HTTP method, headers, and a request body. Changing the payload changes the task without changing the endpoint.</div>

# 2. Inspect and extract the generated text

The raw Responses API result is JSON. Before writing extraction code, inspect its structure. A response normally contains an `output` list; a generated message contains `content`, which contains the output text.

Run the starter code first.

In [ ]:
# Reuse the authentication setup from Unit 1.

payload = {
    "model": "gpt-5.4-nano",
    "instructions": "Answer briefly for a marketing analyst.",
    "input": "What is the difference between reach and conversion rate?",
    "temperature": 0
}

response = requests.post(
    responses_endpoint,
    headers=headers,
    json=payload,
    timeout=60
)
data = response.json()

print(data.keys())
print(data["output"])

### Exercise 2 — Extract only the answer

Modify the starter snippet so that it prints only the model's text answer, rather than the complete JSON response.

In [ ]:
# Starter code — inspect the response before extracting the answer.
payload = {
    "model": "gpt-5.4-nano",
    "instructions": "Answer in one sentence.",
    "input": "What is the difference between reach and conversion rate?",
    "temperature": 0
}

response = requests.post(
    responses_endpoint,
    headers=headers,
    json=payload,
    timeout=60
)
print("Status code:", response.status_code)
data = response.json()
print(json.dumps(data, indent=2))

<details>
<summary><b>Need a hint?</b></summary>

Look for the first output item, then its first content item, then the `text` field. Use the raw response printed by the starter code to verify the path.

</details>

<details>
<summary><b>Solution — Exercise 2</b></summary>

```python
payload = {
    "model": "gpt-5.4-nano",
    "instructions": "Answer in one sentence.",
    "input": "What is the difference between reach and conversion rate?",
    "temperature": 0
}

response = requests.post(
    responses_endpoint,
    headers=headers,
    json=payload,
    timeout=60
)
print("Status code:", response.status_code)

if response.status_code == 200:
    data = response.json()

    def response_text(data):
        for item in data.get("output", []):
            if item.get("type") == "message":
                for content in item.get("content", []):
                    if content.get("type") == "output_text":
                        return content["text"]
        return ""

    print(response_text(data))
else:
    print(json.dumps(response.json(), indent=2))
```

The helper checks the response structure before extracting the generated text.

</details>

<div class="alert alert-block alert-success"><b>Key idea:</b><br>Inspect JSON before extracting from it. The response structure tells you which keys and list positions contain the information you need.</div>

# 3. Add reasoning effort

You can add a `reasoning` field to the payload. It asks the model to spend more or less effort working through the request. It may affect the answer, response time, and cost.

In [ ]:
payload = {
    "model": "gpt-5.4-nano",
    "instructions": "Explain which of these two campaign ideas is more suitable for a budget-conscious customer.",
    "input": "Idea A offers a 10% discount. Idea B offers free delivery.",
    "reasoning": {"effort": "low"}
}

response = requests.post(
    responses_endpoint,
    headers=headers,
    json=payload,
    timeout=60
)
print("Status code:", response.status_code)
print(json.dumps(response.json(), indent=2))

### Exercise 3 — Compare reasoning settings

Run the same request once with `"low"` reasoning effort and once with `"high"` reasoning effort. Compare the answer, response time, or status output.

In [ ]:
# Starter code — change the setting and run the request again.
payload["reasoning"] = {"effort": "low"}

response = requests.post(
    responses_endpoint,
    headers=headers,
    json=payload,
    timeout=60
)
print("Status code:", response.status_code)
print(json.dumps(response.json(), indent=2))

<details>
<summary><b>Need a hint?</b></summary>

Change only `payload["reasoning"]` to `{"effort": "high"}` for the second request.

</details>

<details>
<summary><b>Solution — Exercise 3</b></summary>

```python
for effort in ["low", "high"]:
    payload["reasoning"] = {"effort": effort}
    response = requests.post(
        responses_endpoint,
        headers=headers,
        json=payload,
        timeout=60
    )
    print(f"\nReasoning effort: {effort}")
    print("Status code:", response.status_code)
    print(json.dumps(response.json(), indent=2))
```

Reasoning effort is an optional payload setting. More effort can change the result, response time, or cost.

</details>

<div class="alert alert-block alert-success"><b>Key idea:</b><br>Reasoning effort changes a field in the request payload. It can improve the chance that a complex instruction is followed, but it does not replace validation or causal research design.</div>

# 4. Write a function and use it in a loop

A function packages one API request so that we can reuse it. A loop can then apply that function to every record in a list of dictionaries.

In [ ]:
feedback_items = [
    {
        "response_id": 1,
        "channel": "social",
        "text": "The spring-sale ad made the product feel affordable and relevant."
    },
    {
        "response_id": 2,
        "channel": "email",
        "text": "The retargeting emails arrived too often and annoyed me."
    },
    {
        "response_id": 3,
        "channel": "social",
        "text": "The video clearly explained the product's benefits."
    }
]

def classify_feedback(text):
    # Complete this function so it returns the model's answer.
    pass

### Exercise 4 — Use a function inside a loop

Complete `classify_feedback(text)` so that it sends one piece of feedback to the API and returns the answer. Then use the function inside a loop:

- call `classify_feedback(item["text"])`;
- add the returned value to the same dictionary under `"label"`;
- print each dictionary.

In [ ]:
# Starter code — the function makes one request; the loop reuses it.
def classify_feedback(text):
    payload = {
        "model": "gpt-5.4-nano",
        "instructions": "Return exactly one label: positive, neutral, negative, or unknown.",
        "input": f"Feedback: {text}",
        "temperature": 0
    }
    # Add requests.post() here and return the generated text.
    pass

for item in feedback_items:
    item["label"] = classify_feedback(item["text"])
    print(item)

<details>
<summary><b>Need a hint?</b></summary>

Inside the function, call `requests.post()` with `responses_endpoint`, `headers`, and `json=payload`. If the status code is `200`, return `response_text(response.json())`.

</details>

<details>
<summary><b>Solution — Exercise 4</b></summary>

```python
import time

def response_text(data):
    for output_item in data.get("output", []):
        if output_item.get("type") == "message":
            for content in output_item.get("content", []):
                if content.get("type") == "output_text":
                    return content["text"]
    return ""

def classify_feedback(text):
    payload = {
        "model": "gpt-5.4-nano",
        "instructions": "Return exactly one label: positive, neutral, negative, or unknown.",
        "input": f"Feedback: {text}",
        "temperature": 0
    }
    response = requests.post(
        responses_endpoint,
        headers=headers,
        json=payload,
        timeout=60
    )
    print("Status code:", response.status_code)
    if response.status_code == 200:
        return response_text(response.json()).strip()
    print(json.dumps(response.json(), indent=2))
    return None

feedback_items = [
    {"response_id": 1, "channel": "social", "text": "The spring-sale ad made the product feel affordable and relevant."},
    {"response_id": 2, "channel": "email", "text": "The retargeting emails arrived too often and annoyed me."},
    {"response_id": 3, "channel": "social", "text": "The video clearly explained the product's benefits."}
]

for item in feedback_items:
    item["label"] = classify_feedback(item["text"])
    print(item)
    time.sleep(0.5)
```

The function contains one request. The loop applies that function to each dictionary.

</details>

<div class="alert alert-block alert-success"><b>Key idea:</b><br>Loops turn one successful API request into a repeatable analysis pipeline. A short delay helps avoid unnecessary load on the API.</div>

# 5. Change the instructions

The endpoint, headers, and general request structure can stay the same. To change the task, modify the text in `payload["instructions"]`.

In [ ]:
feedback = "The discount looked attractive, but the landing page was confusing."

payload = {
    "model": "gpt-5.4-nano",
    "instructions": (
        "Return exactly one label: positive, neutral, or negative. "
        "Classify the likely effect of this campaign feedback on purchase intent."
    ),
    "input": f"Feedback: {feedback}",
    "temperature": 0
}

response = requests.post(
    responses_endpoint,
    headers=headers,
    json=payload,
    timeout=60
)
print(response_text(response.json()))

### Exercise 5 — Change the instructions

Modify only `payload["instructions"]` so the model returns:

1. a score from 1 to 5;
2. a one-sentence explanation linked to purchase intent.

The endpoint, headers, and input can stay the same.

In [ ]:
# Starter code — modify the instructions, not the endpoint or headers.
feedback = "The discount looked attractive, but the landing page was confusing."
payload = {
    "model": "gpt-5.4-nano",
    "instructions": "Return exactly one label: positive, neutral, or negative.",
    "input": f"Feedback: {feedback}",
    "temperature": 0
}

response = requests.post(
    responses_endpoint,
    headers=headers,
    json=payload,
    timeout=60
)
print("Status code:", response.status_code)
print(response_text(response.json()))

<details>
<summary><b>Need a hint?</b></summary>

Keep `responses_endpoint`, `headers`, and `payload["input"]` unchanged. Rewrite only `payload["instructions"]` to specify the exact output format.

</details>

<details>
<summary><b>Solution — Exercise 5</b></summary>

```python
def response_text(data):
    for item in data.get("output", []):
        if item.get("type") == "message":
            for content in item.get("content", []):
                if content.get("type") == "output_text":
                    return content["text"]
    return ""

feedback = "The discount looked attractive, but the landing page was confusing."

payload = {
    "model": "gpt-5.4-nano",
    "instructions": (
        "Return a score from 1 to 5 followed by one sentence explaining the "
        "likely effect on purchase intent."
    ),
    "input": f"Feedback: {feedback}",
    "temperature": 0
}

response = requests.post(
    responses_endpoint,
    headers=headers,
    json=payload,
    timeout=60
)
print("Status code:", response.status_code)
if response.status_code == 200:
    print(response_text(response.json()))
else:
    print(json.dumps(response.json(), indent=2))
```

To try the categorical version, replace only the instructions with: `Return exactly one label: positive, neutral, or negative.`

</details>

<div class="alert alert-block alert-success"><b>Key idea:</b><br>The endpoint and authentication can stay fixed while the JSON instructions define different marketing-analysis tasks.</div>

<div class="alert alert-block alert-warning"><b>Using AI APIs responsibly</b><br>
An AI API can automate a coding or enrichment procedure, but it does not automatically make the procedure valid.

<ul>
<li><b>Define the construct:</b> What exactly do labels such as “positive” or “high purchase intent” mean?</li>
<li><b>Test the output:</b> Compare model labels with a small set of human-coded examples.</li>
<li><b>Avoid leakage:</b> Do not include information that would not have been available at the time of the decision.</li>
<li><b>Protect data:</b> Minimize personal or confidential information sent to the API.</li>
<li><b>Document the process:</b> Save the model, prompt, schema, settings, date, and raw outputs.</li>
<li><b>Do not infer causality:</b> A model classification or campaign association does not prove that a campaign caused an outcome.</li>
</ul>

Structured output controls the <b>format</b> of an answer, not whether the answer is substantively correct.
</div>

# 6. Request structured output

When a model response will be stored in a dataset, free-form prose can be difficult to parse. Structured output lets us request a predictable JSON object instead of relying only on instructions such as “return one label.”

For classification, a JSON Schema can also restrict a field to allowed values. This reduces output bleed, but it does not guarantee that the selected label is correct.

The example below classifies campaign feedback. Run the starter request and inspect the returned JSON.

In [ ]:
import json

feedback = "The discount looked attractive, but the landing page was confusing."

payload = {
    "model": "gpt-5.4-nano",
    "instructions": "Classify the likely effect of this campaign feedback on purchase intent.",
    "input": f"Feedback: {feedback}",
    "text": {
        "format": {
            "type": "json_schema",
            "name": "campaign_feedback_classification",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "label": {
                        "type": "string",
                        "enum": ["positive", "neutral", "negative", "unknown"]
                    },
                    "purchase_intent": {
                        "type": "string",
                        "enum": ["high", "medium", "low", "unknown"]
                    },
                    "explanation": {"type": "string"}
                },
                "required": ["label", "purchase_intent", "explanation"],
                "additionalProperties": False
            }
        }
    }
}

response = requests.post(
    responses_endpoint,
    headers=headers,
    json=payload,
    timeout=60
)
print("Status code:", response.status_code)
if response.status_code != 200:
    print(json.dumps(response.json(), indent=2))
else:
    data = response.json()
    structured_text = response_text(data)
    classification = json.loads(structured_text)
    print(classification)
    print("Label:", classification["label"])
    print("Purchase intent:", classification["purchase_intent"])

### Exercise 6 — Adapt the classification schema

Modify the starter request so that it also returns a `channel` field with one of these values:

- `email`
- `social`
- `search`
- `unknown`

Then add a second feedback item and classify it. Try changing the feedback so that `unknown` is the most appropriate label.

In [ ]:
# Starter code — add a field to the JSON Schema below.
classification_schema = {
    "type": "object",
    "properties": {
        "label": {
            "type": "string",
            "enum": ["positive", "neutral", "negative", "unknown"]
        }
    },
    "required": ["label"],
    "additionalProperties": False
}

print(classification_schema)

<details>
<summary><b>Need a hint?</b></summary>

Add the field to `properties` and to `required`. The allowed values belong in the field’s `enum`.

</details>

<details>
<summary><b>Solution — Exercise 6</b></summary>

```python
def response_text(data):
    for item in data.get("output", []):
        if item.get("type") == "message":
            for content in item.get("content", []):
                if content.get("type") == "output_text":
                    return content["text"]
    return ""

import json

feedback = "The discount looked attractive, but the landing page was confusing."
classification_schema = {
    "type": "object",
    "properties": {
        "label": {
            "type": "string",
            "enum": ["positive", "neutral", "negative", "unknown"]
        },
        "purchase_intent": {
            "type": "string",
            "enum": ["high", "medium", "low", "unknown"]
        },
        "explanation": {"type": "string"},
        "channel": {
            "type": "string",
            "enum": ["email", "social", "search", "unknown"]
        }
    },
    "required": ["label", "purchase_intent", "explanation", "channel"],
    "additionalProperties": False
}

payload = {
    "model": "gpt-5.4-nano",
    "instructions": "Classify the feedback and return only the requested JSON object.",
    "input": f"Feedback: {feedback}",
    "text": {
        "format": {
            "type": "json_schema",
            "name": "campaign_feedback_classification",
            "strict": True,
            "schema": classification_schema
        }
    }
}

response = requests.post(
    responses_endpoint,
    headers=headers,
    json=payload,
    timeout=60
)
print("Status code:", response.status_code)
if response.status_code == 200:
    classification = json.loads(response_text(response.json()))
    print(classification)
else:
    print(json.dumps(response.json(), indent=2))
```

The complete schema defines the field types, permitted labels, required fields, and prohibition on additional fields.

</details>

<div class="alert alert-block alert-success"><b>Key idea:</b><br>Structured output makes model results easier to parse and store. It constrains the format, not the truth of the classification.</div>

# 7. Read, enrich, and save JSON Lines

A larger dataset can be stored as **JSON Lines**: one JSON dictionary per line. This uses the same list-of-dictionaries and file-reading ideas you already know.

The workflow is:

```text
read one record → call the function → add a field → write one record
```

In [ ]:
import json

records = [
    {"response_id": 1, "channel": "social", "text": "The ad was clear and relevant."},
    {"response_id": 2, "channel": "email", "text": "The messages arrived too often."}
]

with open("feedback.jsonl", "w") as file:
    for record in records:
        file.write(json.dumps(record) + "\n")

print("Saved feedback.jsonl")

### Exercise 7 — Enrich a JSON Lines file

Read `feedback.jsonl`, use `classify_feedback()` for each record, add the returned value under `"label"`, and save the result as `feedback_enriched.jsonl`.

Test on a small file first: every record creates an API request.

In [ ]:
# Starter code — read one JSON object per line.
import json

records = []
with open("feedback.jsonl", "r") as file:
    for line in file:
        records.append(json.loads(line))

for record in records:
    print(record)

<details>
<summary><b>Need a hint?</b></summary>

After reading the records, loop over them and assign `record["label"] = classify_feedback(record["text"])`. Then open a new file with mode `"w"` and write each dictionary with `json.dumps(record) + "\n"`.

</details>

<details>
<summary><b>Solution — Exercise 7</b></summary>

```python
import json
import time

records = []
with open("feedback.jsonl", "r") as file:
    for line in file:
        records.append(json.loads(line))

for record in records:
    record["label"] = classify_feedback(record["text"])
    time.sleep(0.5)

with open("feedback_enriched.jsonl", "w") as file:
    for record in records:
        file.write(json.dumps(record) + "\n")

print("Saved feedback_enriched.jsonl")
for record in records:
    print(record)
```

The original fields remain in each dictionary, and the API-generated `label` is added before the records are saved again.

</details>

<div class="alert alert-block alert-success"><b>Key idea:</b><br>AI enrichment follows the familiar workflow: read records, call a function, add a field, and save the records again.</div>

<div class="alert alert-block alert-warning"><b>Before processing a larger dataset</b><br>

An AI API can automate a coding or enrichment procedure, but it does not automatically make the procedure valid.

<ul>
<li><b>Define the construct:</b> What exactly does each label or score mean? Include an <code>unknown</code> option when appropriate.</li>
<li><b>Use Python for calculations:</b> Ask the model to extract or classify information, but calculate totals, averages, rates, and comparisons with Python.</li>
<li><b>Test the output:</b> Compare a small sample with human-coded examples and inspect disagreements.</li>
<li><b>Estimate requests and cost:</b> Test on a few records before starting a larger loop.</li>
<li><b>Protect data:</b> Remove unnecessary personal or confidential information.</li>
<li><b>Document the process:</b> Save the model, endpoint, prompt, schema, settings, date, and raw outputs.</li>
</ul>

Structured output controls the <b>format</b> of an answer, not whether the answer is substantively correct. A generated classification or campaign association does not prove causality.
</div>

# Wrap-up

You have now used the HTTP layer directly to:

- Authenticate with a Bearer token.
- Send JSON payloads to `/responses` with `requests.post()`.
- Inspect status codes and parse JSON responses.
- Add an optional reasoning setting.
- Package one request in a function and reuse it in a loop.
- Request structured output and prepare results for validation.
- Read, enrich, and save JSON Lines records.

The Python SDK can wrap many of these details, but `requests` makes the underlying API mechanics visible.

## Appendix — Python SDKs

You have used `requests` directly so the endpoint, headers, payload, and response are visible. A **software development kit (SDK)** is a library that wraps these HTTP requests and offers convenient Python functions.

For example, an OpenAI-compatible SDK can send the same kind of request:

```python
from openai import OpenAI

client = OpenAI(
    api_key=api_key,
    base_url="https://api.tilburg.ai/v1"
)

response = client.responses.create(
    model="gpt-5.4-nano",
    input="Which customer segment is most likely to respond to a discount campaign?"
)

print(response.output_text)
```

The SDK does not use a fundamentally different communication mechanism. APIs still receive HTTP requests; the SDK constructs them and parses the response for you. Use `requests` when you need to see or customize HTTP details, and use an SDK when its helper functions reduce boilerplate.